# 모듈 4. SQL/Python Tool 최소 권한

기존 강의자료 `1. langchain/6. Tools_Agents/4-1. SQLite3 with QuerySQLDatabaseTool.ipynb`와 `2. Python Tools.ipynb`의 SQL/Python Tool 흐름에 최소 권한 가드레일을 추가합니다.

## A/B 테스트 구조

- **A: 기존 코드 그대로 사용** - `QuerySQLDatabaseTool`과 Python 실행 Tool을 그대로 노출합니다.
- **B: 기존 코드에 가드레일 추가** - SQL은 SELECT만 허용하고 Python은 위험 패턴과 실행 시간을 제한합니다.

In [ ]:
import concurrent.futures
import os
import sqlite3
import subprocess
import sys
from pathlib import Path

from dotenv import load_dotenv
from langchain_community.tools.sql_database.tool import QuerySQLDatabaseTool
from langchain_community.utilities import SQLDatabase
from langchain_core.tools import StructuredTool, tool


load_dotenv()

DB_PATH = Path(os.getenv("SECURITY_DEMO_SQLITE_PATH", "security_demo.sqlite"))


def create_sample_database() -> None:
    conn = sqlite3.connect(DB_PATH)
    conn.executescript(
        '''
        DROP TABLE IF EXISTS employees;
        CREATE TABLE employees (
            id INTEGER PRIMARY KEY,
            name TEXT NOT NULL,
            department TEXT NOT NULL,
            salary INTEGER NOT NULL
        );
        INSERT INTO employees (name, department, salary) VALUES
            ('Alice', 'Data', 7000),
            ('Bob', 'Backend', 6500),
            ('Carol', 'Security', 7200);
        '''
    )
    conn.close()


def create_sql_tool() -> QuerySQLDatabaseTool:
    db = SQLDatabase.from_uri(f"sqlite:///{DB_PATH.as_posix()}")
    return QuerySQLDatabaseTool(db=db)

## 1단계: A - 기존 SQL Tool 그대로 사용

기존 SQLite 예제처럼 샘플 DB를 만들고 `QuerySQLDatabaseTool`을 바로 사용합니다. 조회에는 편하지만, 쓰기/삭제 쿼리도 Tool까지 도달할 수 있습니다.

In [ ]:
create_sample_database()
sql_tool = create_sql_tool()

print("정상 조회:")
print(sql_tool.invoke({"query": "SELECT name, department, salary FROM employees"}))

print("\n위험 쿼리:")
print(sql_tool.invoke({"query": "DROP TABLE employees"}))

print("\nDROP 이후 조회:")
print(sql_tool.invoke({"query": "SELECT * FROM employees"}))

### SQL A안 문제점 확인

A안에서 실제로 어떤 SQL 사고가 발생했는지 확인합니다.

In [ ]:
table_check_after_drop = sql_tool.invoke({"query": "SELECT COUNT(*) FROM employees"})
table_was_dropped = "no such table" in table_check_after_drop.lower()

print("SQL A안 보안 진단")
print("- DROP 이후 조회 결과:", table_check_after_drop)
print("- employees 테이블 삭제됨:", table_was_dropped)
if table_was_dropped:
    print("=> 문제: SQL Tool이 조회용이어야 하는데 DDL 명령까지 실행했습니다.")

## 2단계: B - 기존 SQL Tool 앞에 SELECT 전용 Guardrail 추가

`QuerySQLDatabaseTool`은 그대로 사용하되, 실행 전 검증 함수를 둡니다. 실무에서는 이 코드와 함께 읽기 전용 DB 계정도 사용해야 합니다.

In [ ]:
forbidden_sql_keywords = ["drop", "delete", "update", "insert", "alter", "truncate", "create"]


def validate_select_only(query: str) -> tuple[bool, str]:
    normalized = " ".join(query.lower().split())
    if not normalized.startswith("select "):
        return False, "SELECT 문만 허용됩니다."
    if ";" in normalized.rstrip(";"):
        return False, "여러 SQL 문장을 한 번에 실행할 수 없습니다."
    if any(keyword in normalized for keyword in forbidden_sql_keywords):
        return False, "쓰기/삭제/DDL 키워드는 허용되지 않습니다."
    return True, "허용"


def safe_sql_query(query: str) -> str:
    allowed, reason = validate_select_only(query)
    if not allowed:
        return f"거부: {reason}"
    return sql_tool.invoke({"query": query})


create_sample_database()
sql_tool = create_sql_tool()

safe_sql_tool = StructuredTool.from_function(
    func=safe_sql_query,
    name="safe_sql_query",
    description="SELECT 전용 SQL 조회 Tool",
)

print("위험 쿼리:", safe_sql_tool.invoke({"query": "DROP TABLE employees"}))
print("정상 쿼리:", safe_sql_tool.invoke({"query": "SELECT name, salary FROM employees ORDER BY salary DESC"}))

### SQL B안 개선 확인

B안에서 SQL Guardrail이 위험 쿼리를 막고 정상 조회는 허용하는지 확인합니다.

In [ ]:
blocked_drop = safe_sql_tool.invoke({"query": "DROP TABLE employees"})
table_check_after_guard = safe_sql_tool.invoke({"query": "SELECT COUNT(*) FROM employees"})

print("SQL B안 보안 진단")
print("- DROP 차단 결과:", blocked_drop)
print("- 테이블 유지 확인:", table_check_after_guard)
print("- 위험 쿼리 차단:", blocked_drop.startswith("거부"))
print("=> 개선: 기존 SQL Tool을 유지하면서 실행 전 정책으로 파괴적 쿼리를 막습니다.")

## 3단계: A - 기존 Python Tool 그대로 사용

기존 Python Tool 예제처럼 코드를 받아 `python -c`로 실행합니다. 이 구조는 편하지만, 파일 접근이나 시스템 명령 실행으로 확장될 수 있어 매우 조심해야 합니다.

In [ ]:
@tool
def python_tool(code: str) -> str:
    """Python 코드를 실행하고 결과를 반환한다."""
    result = subprocess.run(
        [sys.executable, "-c", code],
        capture_output=True,
        text=True,
        timeout=3,
    )
    return result.stdout or result.stderr


print(python_tool.invoke({"code": "print(2**10)"}))
print(python_tool.invoke({"code": "import os; print(os.getcwd())"}))

### Python A안 문제점 확인

A안에서 Python Tool이 실행 환경 정보까지 읽을 수 있는지 확인합니다.

In [ ]:
unsafe_python_result = python_tool.invoke({"code": "import os; print(os.getcwd())"})
runtime_info_exposed = bool(unsafe_python_result.strip())

print("Python A안 보안 진단")
print("- 런타임 정보 출력:", unsafe_python_result.strip())
print("- 파일/시스템 접근 가능 코드가 실행됨:", runtime_info_exposed)
if runtime_info_exposed:
    print("=> 문제: Python Tool이 분석 계산을 넘어 실행 환경 정보까지 노출했습니다.")

## 4단계: B - Python Tool에 최소 권한 Guardrail 추가

기존 Python Tool의 목적은 유지하되, 위험 키워드와 실행 시간을 제한합니다. 운영에서는 별도 컨테이너, 네트워크 차단, 파일시스템 제한까지 함께 필요합니다.

In [ ]:
forbidden_python_patterns = [
    "import os",
    "import subprocess",
    "open(",
    "socket",
    "eval(",
    "exec(",
    "__",
]


def run_python_with_timeout(code: str, timeout_seconds: float = 1.0) -> str:
    def _run() -> str:
        namespace = {"answer": None}
        exec(code, {"__builtins__": {"len": len, "sum": sum, "range": range}}, namespace)
        return str(namespace.get("answer"))

    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as executor:
        future = executor.submit(_run)
        return future.result(timeout=timeout_seconds)


def safe_python_exec(code: str) -> str:
    normalized = code.replace(" ", "").lower()
    for pattern in forbidden_python_patterns:
        if pattern.replace(" ", "").lower() in normalized:
            return f"거부: 금지 패턴 감지 - {pattern}"
    try:
        return run_python_with_timeout(code)
    except concurrent.futures.TimeoutError:
        return "거부: 실행 시간이 너무 깁니다."
    except Exception as exc:
        return f"실행 오류: {exc}"


safe_python_tool = StructuredTool.from_function(
    func=safe_python_exec,
    name="safe_python_exec",
    description="제한된 Python 코드만 실행하는 Tool",
)

print("위험 코드:", safe_python_tool.invoke({"code": "import os; answer = os.getcwd()"}))
print("정상 코드:", safe_python_tool.invoke({"code": "answer = sum(range(10))"}))

### Python B안 개선 확인

B안에서 위험 Python 코드는 차단하고 정상 계산은 허용하는지 확인합니다.

In [ ]:
blocked_python = safe_python_tool.invoke({"code": "import os; answer = os.getcwd()"})
allowed_python = safe_python_tool.invoke({"code": "answer = sum(range(10))"})

print("Python B안 보안 진단")
print("- 위험 코드 차단 결과:", blocked_python)
print("- 정상 계산 결과:", allowed_python)
print("- 위험 코드 차단:", blocked_python.startswith("거부"))
print("=> 개선: 분석 계산은 허용하고 파일/시스템 접근성 코드는 차단합니다.")

## 정리

기존 SQL/Python Tool 코드는 강력하기 때문에 그대로 운영 Agent에 노출하면 위험합니다. 기존 Tool 사용 방식은 유지하되, 실행 직전에 최소 권한 정책을 코드로 강제해야 합니다.